# 09. Классы для создания нейросетей

Видео: [9. Классы для создания нейросетей](https://www.youtube.com/watch?v=cCg2mc-O47k&list=PLBP4Q3FNSLK2rtGPBsK-aMAetYj-8yg_1&index=10)

## О чём урок

В видео 8 ты уже знаешь кубики: `Linear`, ReLU, loss, Adam. Здесь — **как собрать из них модель**.

| способ | когда |
|:---|:---|
| `nn.Sequential` | слои строго друг за другом, один вход → один выход |
| свой класс `nn.Module` | свой `forward`: несколько входов/выходов, ветки |
| `nn.ModuleList` | список слоёв, который видит оптимизатор |
| `nn.ModuleDict` | словарь модулей (например, выбор активации по ключу) |

> **Конспект:** тело MLP — [`d.4.1`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.1 Полносвязные нейронные сети/d.4.1 Полносвязные нейронные сети.md) **§5–7**; активации между слоями — **§8–9**. Карта курса — [`0_как_учить_pytorch.md`](../0_как_учить_pytorch.md).

**Как смотреть.** Видео ~13 мин — **кусками** по таблице → пауза → блок.

| блок | о чём | где roughly |
|:---|:---|:---|
| 1 | `Sequential` | начало |
| 2 | свой `nn.Module` | середина |
| 3 | два входа / два выхода | автопилот, детекция |
| 4 | `ModuleList` vs обычный `list` | конец |
| 5 | `ModuleDict` | конец |

**Как решать.** Замени `...` на код → проверка. Решения — `09_классы_нейросетей_решения.ipynb` только после попытки.


In [91]:
import torch
import torch.nn as nn

torch.manual_seed(0)


def need(cond, msg):
    assert cond, msg


def need_tensor(x, name):
    assert isinstance(x, torch.Tensor), f"{name} — пока не тензор (замени ... на код)"


print("torch", torch.__version__)


torch 2.14.1+cpu


## 1. <span style="color:#7CDBA0">nn.Sequential</span> — слои друг за другом

> **Конспект:** цепочка Linear → ReLU → Linear — [`d.4.1`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.1 Полносвязные нейронные сети/d.4.1 Полносвязные нейронные сети.md) **§7**, **§8**.

### Зачем

В видео 8 кубики вызывались по отдельности: `layer_2(relu(layer_1(x)))`.  
`Sequential` **склеивает** их в один объект-модель: `model(x)` сам прогоняет всё по порядку.

Это удобно, когда архитектура **прямолинейная**: один тензор входит → проходит слой за слоем → один тензор выходит (FashionMNIST как раз такой случай).

### Что делает задание 1.1

`784 → 128 → ReLU → 10`:
- **784** — пиксели 28×28, сплющенные в вектор (один пример);
- **128** — скрытый слой (новые признаки, твой §6–7);
- **ReLU** — чтобы два Linear не схлопнулись в один (§8);
- **10** — логиты классов (ещё не softmax: при обучении с CE softmax внутри loss).

Батч `(32, 784)` → на выходе `(32, 10)`: 32 картинки, по 10 чисел на каждую.

### Когда Sequential мало

- несколько входов или выходов;
- нужно вернуть активацию «из середины»;
- ветвление / skip (позже ResNet).

Тогда — свой класс `nn.Module` (блок 2–3).


In [92]:
# 1.1 Sequential: 784 → 128 → ReLU → 10 (как в видео 8)
model_seq = nn.Sequential(
    nn.Linear(784,128),
    nn.ReLU(),
    nn.Linear(128, 10)
)
x = torch.randn(32, 784)
out_seq = model_seq(x)
print("Sequential out:", tuple(out_seq.shape) if isinstance(out_seq, torch.Tensor) else out_seq)
print(model_seq)



Sequential out: (32, 10)
Sequential(
  (0): Linear(in_features=784, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=10, bias=True)
)


In [93]:
need(isinstance(model_seq, nn.Sequential), "model_seq = nn.Sequential(...)")
need_tensor(out_seq, "out_seq")
need(tuple(out_seq.shape) == (32, 10), f"ожидали (32, 10), получили {tuple(out_seq.shape)}")
need(len(list(model_seq.children())) == 3, "три модуля: Linear, ReLU, Linear")
print("1.1 — верно")


1.1 — верно


### Задание 1.2 — имена слоёв через `add_module`

### Зачем имена

В `nn.Sequential(nn.Linear(...), nn.ReLU(), nn.Linear(...))` дети называются `"0"`, `"1"`, `"2"`.  
Работать можно (`model[2]`), но через месяц непонятно, что за «двойка».

`add_module("layer_1", ...)` даёт:
- обращение `model.layer_1`, `model.act`, `model.layer_2`;
- в `state_dict` ключи вида `layer_1.weight`, а не `0.weight` — удобнее при сохранении и отладке;
- проще точечно заменить один слой.

На качество обучения имена **не влияют** — только читаемость и удобство.  
Та же сеть, что в 1.1, просто с человеческими ярлыками.


In [94]:
# 1.2 Имена слоёв через add_module
model_named = nn.Sequential()

# добавь layer_1, act, layer_2 с именами
model_named.add_module("layer_1", nn.Linear(784, 128))
model_named.add_module("act", nn.ReLU())
model_named.add_module("layer_2", nn.Linear(128, 10))
print(list(model_named.named_children()) if hasattr(model_named, "named_children") else model_named)


[('layer_1', Linear(in_features=784, out_features=128, bias=True)), ('act', ReLU()), ('layer_2', Linear(in_features=128, out_features=10, bias=True))]


In [95]:
names = [n for n, _ in model_named.named_children()]
need(names == ["layer_1", "act", "layer_2"], f"имена слоёв: {names}")
need(isinstance(model_named.layer_1, nn.Linear) and model_named.layer_1.in_features == 784, "layer_1 = Linear(784, ...)")
need(isinstance(model_named.act, nn.ReLU), "act = ReLU")
need(isinstance(model_named.layer_2, nn.Linear) and model_named.layer_2.out_features == 10, "layer_2 → 10")
print("1.2 — верно")


1.2 — верно


## 2. Свой класс <span style="color:#C9A0DC">nn.Module</span>

> **Конспект:** [`d.4.1`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.1 Полносвязные нейронные сети/d.4.1 Полносвязные нейронные сети.md) **§5–7** — что такое слой/MLP; здесь — **как это оформить в коде PyTorch**.

### Зачем класс, если есть Sequential

`Sequential` = «строго труба».  
Свой класс нужен, когда **ты решаешь**, как данные идут: два входа, два выхода, цикл по слоям, условие, сумма веток (блок 3–4).

Даже для простой сети класс — нормальный стиль проекта: всё в одном месте (`FashionMLP`), параметры видны через `model.parameters()`, потом легко отдать в Adam.

### Правила (обязательно)

1. Наследование: `class MLP(nn.Module)`.
2. В `__init__` **первой** строкой `super().__init__()` — подключается учёт слоёв у родителя.
3. Слои пиши в `self`: `self.layer_1 = nn.Linear(...)` — так они **регистрируются** (попадут в `parameters()` / `state_dict`).
4. В `forward(self, x)` — прямой проход: как в конспекте «×W+b → ReLU → …».
5. Вызывай `model(x)`, не `model.forward(x)` вручную — так срабатывают служебные хуки PyTorch.

### Что проверяем в 2.1

Та же архитектура, что Sequential: 784→128→ReLU→10, **101770** параметров (как считали в уроке 8).  
Смысл: два способа записи — одна математика.


In [96]:
# 2.1 Свой класс: та же сеть 784 → 128 → ReLU → 10
class MLP(nn.Module):
    def __init__(self, in_features=784, hidden=128, out_features=10):
        super().__init__()
        self.layer_1 = nn.Linear(in_features, hidden)
        self.act = nn.ReLU()
        self.layer_2 = nn.Linear(hidden, out_features)

    def forward(self, x):
        x = self.layer_1(x)
        x = self.act(x)
        x = self.layer_2(x)
        return x



torch.manual_seed(0)
mlp = MLP()
x = torch.randn(32, 784)
out_mlp = mlp(x)
print("MLP out:", tuple(out_mlp.shape) if isinstance(out_mlp, torch.Tensor) else out_mlp)


MLP out: (32, 10)


In [97]:
need(isinstance(mlp, MLP), "mlp = MLP()")
need(isinstance(mlp, nn.Module), "MLP должен наследоваться от nn.Module")
need_tensor(out_mlp, "out_mlp")
need(tuple(out_mlp.shape) == (32, 10), f"shape={tuple(out_mlp.shape)}")
need(hasattr(mlp, "layer_1") and hasattr(mlp, "layer_2"), "зарегистрируй слои как self.layer_1 / self.layer_2")
n = sum(p.numel() for p in mlp.parameters())
need(n == 101770, f"параметров как в уроке 8: 101770, у тебя {n}")
print("2.1 — верно")


2.1 — верно


## 3. Несколько входов и выходов

> **Конспект:** одна «голова» или несколько выходов под задачу — идея из [`d.4.1`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.1 Полносвязные нейронные сети/d.4.1 Полносвязные нейронные сети.md) **§10** (здесь без новых формул loss).

### Почему Sequential не подходит

У `Sequential` контракт жёсткий: **один** аргумент → **один** тензор.  
Автопилот из видео (камера + скорость + руль) или детекция (класс + рамка) так не собрать.

### Задание 3.1 — два входа

- `x` — основной поток признаков, идёт в `Linear`;
- `y` — второй тензор **той же ширины**, что выход этого Linear;
- складываем `layer_1(x) + y` (в видео — сумма, **не** склейка `cat`);
- дальше ReLU и второй Linear.

Зачем так в курсе: показать, что `forward(self, x, y)` — обычный Python, ты волен смешивать тензоры как угодно (лишь бы формы совпали).

### Задание 3.2 — два выхода

Возвращаем кортеж `(logits, h)`:
- `logits` — то, что обычно идёт в loss;
- `h` — промежуточные признаки после ReLU (для отладки, второй головы, визуализации).

В реальных задачах «несколько голов» часто считают **несколько loss** и суммируют — это уже видео про обучение; сейчас важно уметь **вернуть** оба тензора из `forward`.


In [98]:
# 3.1 Два входа: layer_1(x) + y → ReLU → layer_2
class TwoInputs(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(4, 3)
        self.act = nn.ReLU()
        self.layer_2 = nn.Linear(3, 2)

    def forward(self, x, y):
        # y той же формы, что выход layer_1: (B, 3)
        x = self.layer_1(x)
        x = self.act(x+y)
        out = self.layer_2(x)
        return out


torch.manual_seed(0)
two_in = TwoInputs()
xb = torch.randn(5, 4)
yb = torch.randn(5, 3)
out_two = two_in(xb, yb)
print("two inputs out:", tuple(out_two.shape))


two inputs out: (5, 2)


In [99]:
need(isinstance(two_in, TwoInputs), "two_in = TwoInputs()")
need_tensor(out_two, "out_two")
need(tuple(out_two.shape) == (5, 2), f"shape={tuple(out_two.shape)}")
# проверка, что y реально участвует: без y forward не вызвать
need(
    "y" in two_in.forward.__code__.co_varnames
    or two_in.forward.__code__.co_argcount >= 3,
    "forward(self, x, y)",
)
print("3.1 — верно")


3.1 — верно


In [100]:
# 3.2 Два выхода: финальные логиты + промежуточный h после ReLU
class TwoOutputs(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(4, 3)
        self.act = nn.ReLU()
        self.layer_2 = nn.Linear(3, 2)

    def forward(self, x):
        h = self.act(self.layer_1(x))
        logits = self.layer_2(h)
        return logits, h


torch.manual_seed(0)
two_out = TwoOutputs()
logits, h = two_out(torch.randn(5, 4))
print("logits:", tuple(logits.shape), "h:", tuple(h.shape))


logits: (5, 2) h: (5, 3)


In [101]:
need(isinstance(two_out, TwoOutputs), "two_out = TwoOutputs()")
need_tensor(logits, "logits")
need_tensor(h, "h")
need(tuple(logits.shape) == (5, 2) and tuple(h.shape) == (5, 3), f"logits {tuple(logits.shape)}, h {tuple(h.shape)}")
need(bool((h >= 0).all()), "h после ReLU: все ≥ 0")
print("3.2 — верно")


3.2 — верно


## 4. <span style="color:#E6C07B">nn.ModuleList</span>

> **Конспект:** глубина сети — [`d.4.1`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.1 Полносвязные нейронные сети/d.4.1 Полносвязные нейронные сети.md) **§7**. Здесь — ловушка Python + PyTorch: **кто видит веса**.

### Зачем

Когда слоёв много и они однотипные, их собирают **циклом** в список, а не копипастят `self.layer_1`, `self.layer_2`, …  
Но обычный `list` для PyTorch — «чужой ящик»: слои там есть, а в `model.parameters()` — **нет**.

### Почему это критично

`Adam(model.parameters())` обновляет только зарегистрированные веса.  
Положил `Linear` в обычный `list` → forward может даже что-то считать, но **обучение не двигает эти веса** (или их «нет» в отчёте параметров). Ошибка тихая и очень неприятная.

`nn.ModuleList([...])` — список, который родительский `Module` **регистрирует**.

### Связь с видео

Автор строит длинную сеть циклом и забирает промежуточные выходы (`out_1`, `out_2`).  
`Sequential` так не умеет → `ModuleList` + свой `forward` с циклом и условиями.


In [102]:
# 4.1 Обычный list vs ModuleList: кто в parameters()
class BadListNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = [nn.Linear(4, 3), nn.ReLU(), nn.Linear(3, 2)]  # НЕ зарегистрированы

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


class GoodListNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(4, 3), nn.ReLU(), nn.Linear(3, 2)])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


bad = BadListNet()
good = GoodListNet()
n_bad = sum(p.numel() for p in bad.parameters())
n_good = sum(p.numel() for p in good.parameters())
print("параметров bad (list):", n_bad)
print("параметров good (ModuleList):", n_good)
# Linear(4,3)=15 + Linear(3,2)=8 → 23


параметров bad (list): 0
параметров good (ModuleList): 23


In [103]:
need(n_bad == 0, f"у BadListNet в parameters() должно быть 0, а не {n_bad} — слои в обычном list не регистрируются")
need(n_good == 23, f"GoodListNet: Linear(4,3)+Linear(3,2) = 15+8 = 23, у тебя {n_good}")
need(isinstance(good.layers, nn.ModuleList), "self.layers = nn.ModuleList([...])")
need(isinstance(bad.layers, list), "у BadListNet оставь обычный list")
yb = good(torch.randn(5, 4))
need_tensor(yb, "выход good")
need(tuple(yb.shape) == (5, 2), f"good out {tuple(yb.shape)}")
print("4.1 — верно")


4.1 — верно


### Задание 4.2 — цикл слоёв и промежуточные выходы

### Что сделать

Ширины `8 → 6 → 4 → 2`:
- между Linear ставь ReLU;
- **после последнего** Linear ReLU не ставь (как «голова» без активации / под регрессию или логиты).

Если `collect_linear=True` — верни `(финальный_x, [выходы каждого Linear])`.  
Так ты руками повторяешь идею автора «достать середину сети», только с простым правилом вместо запутанных `i % 4` из ролика.

### Зачем это в проекте потом

Иногда логируют активации, цепляют доп. голову к промежуточному слою или отлаживают «где всё обнулилось».  
Базовый FashionMLP так усложнять не нужно — достаточно знать приём.


In [104]:
# 4.2 ModuleList в цикле + промежуточные выходы после каждого Linear
class DeepModuleList(nn.Module):
    def __init__(self, widths=(8, 6, 4, 2)):
        super().__init__()
        layers = []
        for i in range(len(widths) - 1):
            layers.append(nn.Linear(widths[i], widths[i + 1]))
            if i < len(widths) - 2:  # после последнего Linear ReLU не ставим
                layers.append(nn.ReLU())
        self.layers = nn.ModuleList(layers)

    def forward(self, x, collect_linear=False):
        outs = []
        for layer in self.layers:
            x = layer(x)
            if collect_linear and isinstance(layer, nn.Linear):
                outs.append(x)
        if collect_linear:
            return x, outs
        return x


torch.manual_seed(0)
deep = DeepModuleList()
x = torch.randn(5, 8)
y, mids = deep(x, collect_linear=True)
print(
    "final:",
    tuple(y.shape) if isinstance(y, torch.Tensor) else y,
    "n mid Linear outs:",
    len(mids) if isinstance(mids, list) else mids,
)


final: (5, 2) n mid Linear outs: 3


In [105]:
need(isinstance(deep.layers, nn.ModuleList), "слои в ModuleList")
need_tensor(y, "y")
need(isinstance(mids, list) and len(mids) == 3, f"три выхода Linear (8→6, 6→4, 4→2), сейчас {mids!r}")
need(tuple(y.shape) == (5, 2), f"final {tuple(y.shape)}")
need(tuple(mids[0].shape) == (5, 6) and tuple(mids[-1].shape) == (5, 2), "формы промежуточных Linear")
print("4.2 — верно")


4.2 — верно


## 5. <span style="color:#6CB6FF">nn.ModuleDict</span>

### Зачем

`ModuleDict` — словарь **зарегистрированных** модулей: ключ → слой.  
Как `ModuleList`, только доступ по имени (`"relu"` / `"tanh"`), а не по индексу.

В задании при создании сети выбираешь активацию:

```python
self.acts = nn.ModuleDict({"relu": nn.ReLU(), "tanh": nn.Tanh()})
...
x = self.acts[self.act_name](x)
```

### Когда пригодится

- быстро сменить ReLU на другую активацию без копипасты `forward`;
- несколько вариантов блока (разные нормы / активации) под гиперпараметр;
- читать конфиг: `act_name="tanh"` и собрать сеть.

Обычный `dict` с модулями — та же ловушка, что `list`: могут не попасть в `parameters()`. Для слоёв — `ModuleDict`.


In [107]:
# 5.1 ModuleDict: выбор активации по ключу
class ActChoiceNet(nn.Module):
    def __init__(self, act_name="relu"):
        super().__init__()
        self.layer_1 = nn.Linear(4, 3)
        self.acts = nn.ModuleDict(
            {
                "relu": nn.ReLU(),
                "tanh": nn.Tanh(),
            }
        )
        self.act_name = act_name
        self.layer_2 = nn.Linear(3, 2)

    def forward(self, x):
        x = self.layer_1(x)
        x = self.acts[self.act_name](x)
        return self.layer_2(x)


torch.manual_seed(0)
net_relu = ActChoiceNet("relu")
torch.manual_seed(0)
net_tanh = ActChoiceNet("tanh")
# одинаковый seed + одинаковый layer_1 init? разные объекты — веса разные.
# Сравним тип активации и что forward работает:
xr = torch.randn(5, 4)
print("relu out", tuple(net_relu(xr).shape))
print("tanh out", tuple(net_tanh(xr).shape))
print("ключи acts:", list(net_relu.acts.keys()))


relu out (5, 2)
tanh out (5, 2)
ключи acts: ['relu', 'tanh']


In [108]:
need(isinstance(net_relu.acts, nn.ModuleDict), "self.acts = nn.ModuleDict({{...}})")
need(set(net_relu.acts.keys()) == {"relu", "tanh"}, f"ключи: {list(net_relu.acts.keys())}")
need(net_relu.act_name == "relu" and net_tanh.act_name == "tanh", "act_name должен совпадать с аргументом")
need_tensor(net_relu(xr), "forward relu")
need_tensor(net_tanh(xr), "forward tanh")
print("5.1 — верно")


5.1 — верно


## 6. Что мы сделали в этом уроке

1. **`Sequential`** — быстрая «труба» один вход → один выход; имена через `add_module` для читаемости.
2. **Свой `nn.Module`** — `__init__` регистрирует слои, `forward` задаёт путь данных (основа проекта).
3. **Несколько входов/выходов** — только свой `forward` (сумма веток, возврат кортежа); `Sequential` так не умеет.
4. **`ModuleList`** — список слоёв для цикла **и** для оптимизатора; обычный `list` веса «прячет».
5. **`ModuleDict`** — то же для словаря (выбор активации по ключу).

**Правило выбора:** простая цепочка → `Sequential` или тонкий класс с `Sequential` внутри; любая нестандартная схема → свой `forward`.

Дальше видео **10**: методы модели + сборка под задачу. У нас — класс в `project/fashion_mnist.ipynb` (раздел «Видео 9»). Теория MLP — [`d.4.1`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.1 Полносвязные нейронные сети/d.4.1 Полносвязные нейронные сети.md) §5–7.
